# 07 — AutoML (AutoKeras): Gaussian Noise Experiment

Same AutoKeras approach, but trained on SCIN images from our GCS bucket.

1. **Noised-only** — trained on Gaussian-noised SCIN images
2. **Original + Noised** — trained on both original and noised SCIN images

In [ ]:
!pip install autokeras tensorflow

In [ ]:
# Authenticate + download images from bucket
from google.colab import auth
auth.authenticate_user()

!mkdir -p data/scin/images data/scin/images_noised
!gcloud storage cp -r -n gs://skin-tone-project/scin_images/originals/ data/scin/images/
!gcloud storage cp -r -n gs://skin-tone-project/scin_images/noised/images_noised/ data/scin/images_noised/

In [ ]:
# Download SCIN metadata CSVs for labels
!mkdir -p data/scin
!gsutil cp gs://dx-scin-public-data/dataset/scin_cases.csv data/scin/
!gsutil cp gs://dx-scin-public-data/dataset/scin_labels.csv data/scin/

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
import tensorflow as tf
import autokeras as ak

# Build labeled dataframe from SCIN metadata
cases = pd.read_csv("data/scin/scin_cases.csv")
labels = pd.read_csv("data/scin/scin_labels.csv")
df = cases.merge(labels, on="case_id", how="inner")
df = df[df["fitzpatrick_skin_type"].str.startswith("FST", na=False)].copy()
df["fitzpatrick_centaur"] = df["fitzpatrick_skin_type"].str.replace("FST", "", regex=False).astype(int) - 1

# Explode image paths (each case can have up to 3 images)
image_cols = [c for c in df.columns if c.startswith("image_") and c.endswith("_path")]
rows = []
for _, row in df.iterrows():
    for col in image_cols:
        if pd.notna(row[col]) and str(row[col]).strip():
            rows.append({"hasher": Path(str(row[col])).stem, "fitzpatrick_centaur": row["fitzpatrick_centaur"]})

df = pd.DataFrame(rows)
df = df.dropna()
print(f"Total labeled images: {len(df)}")
print(df["fitzpatrick_centaur"].value_counts().sort_index())

In [ ]:
IMG_SIZE = 96
MAX_TRAIN_SAMPLES = 5000
BATCH_SIZE = 32
MAX_WORKERS = 32

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['fitzpatrick_centaur']
)

In [ ]:
# Find actual image directories (bucket may nest under subdirs)
orig_images = list(Path("data/scin/images").rglob("*.png")) + list(Path("data/scin/images").rglob("*.jpg"))
noised_images = list(Path("data/scin/images_noised").rglob("*.png")) + list(Path("data/scin/images_noised").rglob("*.jpg"))

# Build lookup: stem -> path
orig_lookup = {p.stem: p for p in orig_images}
noised_lookup = {p.stem: p for p in noised_images}

print(f"Original images found: {len(orig_lookup)}")
print(f"Noised images found:   {len(noised_lookup)}")

In [ ]:
def process_image_local(args):
    path, label = args
    try:
        img = Image.open(path).convert('RGB')
        img = img.resize((IMG_SIZE, IMG_SIZE))
        img_array = np.array(img) / 255.0
        return img_array, label
    except:
        return None


def load_images_parallel(df_subset, lookup, suffix="", max_samples=None):
    if max_samples:
        df_subset = df_subset.head(max_samples)

    tasks = []
    for _, row in df_subset.iterrows():
        stem = str(row['hasher']) + suffix
        if stem in lookup:
            tasks.append((lookup[stem], row['fitzpatrick_centaur']))

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(executor.map(process_image_local, tasks))

    X_list, y_list = [], []
    for r in tqdm(results):
        if r is not None:
            X_list.append(r[0])
            y_list.append(r[1])

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)

In [ ]:
# === Experiment 1: Noised Only ===
print("Loading noised training images...")
X_train_noised, y_train_noised = load_images_parallel(train_df, noised_lookup, suffix="_noised", max_samples=MAX_TRAIN_SAMPLES)
print("Loading noised test images...")
X_test_noised, y_test_noised = load_images_parallel(test_df, noised_lookup, suffix="_noised")

print(f"Training: {X_train_noised.shape}, Test: {X_test_noised.shape}")

In [ ]:
clf_noised = ak.ImageClassifier(
    overwrite=True,
    max_trials=10,
    max_model_size=50_000_000
)

clf_noised.fit(
    X_train_noised,
    y_train_noised,
    epochs=10,
    validation_split=0.2
)

loss_noised, acc_noised = clf_noised.evaluate(X_test_noised, y_test_noised)
print(f"\nNoised-Only Test Accuracy: {acc_noised:.4f}")

model_noised = clf_noised.export_model()
model_noised.save("best_model_noised_only.keras")

In [ ]:
# === Experiment 2: Original + Noised ===
print("Loading original training images...")
X_train_orig, y_train_orig = load_images_parallel(train_df, orig_lookup, max_samples=MAX_TRAIN_SAMPLES)
print("Loading original test images...")
X_test_orig, y_test_orig = load_images_parallel(test_df, orig_lookup)

# Combine original + noised
X_train_both = np.concatenate([X_train_orig, X_train_noised])
y_train_both = np.concatenate([y_train_orig, y_train_noised])

print(f"Combined Training: {X_train_both.shape}, Test (originals): {X_test_orig.shape}")

In [ ]:
clf_both = ak.ImageClassifier(
    overwrite=True,
    max_trials=10,
    max_model_size=50_000_000
)

clf_both.fit(
    X_train_both,
    y_train_both,
    epochs=10,
    validation_split=0.2
)

loss_both, acc_both = clf_both.evaluate(X_test_orig, y_test_orig)
print(f"\nOrig+Noised Test Accuracy: {acc_both:.4f}")

model_both = clf_both.export_model()
model_both.save("best_model_orig_and_noised.keras")

In [ ]:
# Evaluate both models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


def evaluate_model(model, X, y, dataset_name="Dataset"):
    y_pred_probs = model.predict(X)
    y_pred = np.argmax(y_pred_probs, axis=1)

    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y, y_pred, average='weighted')
    f1 = f1_score(y, y_pred, average='weighted')

    print(f"\n=== {dataset_name} Metrics ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")

    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f"{dataset_name} Confusion Matrix")
    plt.savefig(f"{dataset_name}_confusion_matrix.png")
    plt.show()


print("=" * 60)
print("RESULTS COMPARISON")
print("=" * 60)

evaluate_model(model_noised, X_test_noised, y_test_noised, "Noised Only")
evaluate_model(model_both, X_test_orig, y_test_orig, "Original + Noised")

print(f"\nNoised-only accuracy:  {acc_noised:.4f}")
print(f"Orig+Noised accuracy:  {acc_both:.4f}")
print(f"Difference:            {acc_both - acc_noised:+.4f}")